In [1]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "backend").is_dir() and (candidate / "datasets" / "raw" / "CMAPSSData.zip").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate project root containing 'backend/' and 'datasets/raw/CMAPSSData.zip'. "
        f"Searched upwards from {current}"
    )

PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "datasets" / "raw"
EXTRACT_DIR = RAW_DATA_DIR / "extracted"
PROCESSED_DATA_DIR = PROJECT_ROOT / "datasets" / "processed"
DOCS_DIR = PROJECT_ROOT / "ml" / "docs"
ZIP_PATH = RAW_DATA_DIR / "CMAPSSData.zip"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = [
    "train_FD001.txt",
    "test_FD001.txt",
    "RUL_FD001.txt",
]

missing_before = [f for f in EXPECTED_FILES if not (EXTRACT_DIR / f).is_file()]

if missing_before:
    print(f"Extracting raw archive: {ZIP_PATH} -> {EXTRACT_DIR}")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    
    missing_after = [f for f in EXPECTED_FILES if not (EXTRACT_DIR / f).is_file()]
    if missing_after:
        raise FileNotFoundError(
            f"Extraction completed but the following expected files are missing from {EXTRACT_DIR}: {missing_after}"
        )
    print("Dataset extracted successfully to:", EXTRACT_DIR)
else:
    print("All expected dataset files already exist in:", EXTRACT_DIR)

print("Project root found at:", PROJECT_ROOT)
print(sorted([f.name for f in EXTRACT_DIR.iterdir()]))


Extracting raw archive: C:\Users\Vivek Nigam\Desktop\YantraDrishti\datasets\raw\CMAPSSData.zip -> C:\Users\Vivek Nigam\Desktop\YantraDrishti\datasets\raw\extracted
Dataset extracted successfully to: C:\Users\Vivek Nigam\Desktop\YantraDrishti\datasets\raw\extracted
Project root found at: C:\Users\Vivek Nigam\Desktop\YantraDrishti
['Damage Propagation Modeling.pdf', 'RUL_FD001.txt', 'RUL_FD002.txt', 'RUL_FD003.txt', 'RUL_FD004.txt', 'readme.txt', 'test_FD001.txt', 'test_FD002.txt', 'test_FD003.txt', 'test_FD004.txt', 'train_FD001.txt', 'train_FD002.txt', 'train_FD003.txt', 'train_FD004.txt']


In [2]:
import pandas as pd
import numpy as np

columns = (
    ['unit_number', 'cycle',
     'setting_1', 'setting_2', 'setting_3']
    + [f'sensor_{i:02d}' for i in range(1, 22)]
)

train_df = pd.read_csv(
    EXTRACT_DIR / 'train_FD001.txt',
    sep=r'\s+',
    header=None,
    names=columns
)

test_df = pd.read_csv(
    EXTRACT_DIR / 'test_FD001.txt',
    sep=r'\s+',
    header=None,
    names=columns
)

test_rul = pd.read_csv(
    EXTRACT_DIR / 'RUL_FD001.txt',
    sep=r'\s+',
    header=None,
    names=['RUL']
)

print("TRAIN SHAPE:", train_df.shape)
print("TEST SHAPE:", test_df.shape)
print("TEST RUL SHAPE:", test_rul.shape)

display(train_df.head())


TRAIN SHAPE: (20631, 26)
TEST SHAPE: (13096, 26)
TEST RUL SHAPE: (100, 1)


,unit_number,cycle,setting_1,setting_2,setting_3,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [3]:
max_cycles = train_df.groupby('unit_number')['cycle'].max()
train_df['RUL'] = (
    train_df['unit_number'].map(max_cycles) - train_df['cycle']
)

print("Training RUL generated!")

display(
    train_df[['unit_number', 'cycle', 'RUL']].head(10)
)

print("\nRUL Statistics:")
display(train_df['RUL'].describe())

Training RUL generated!

RUL Statistics:


,unit_number,cycle,RUL
0,1,1,191
1,1,2,190
2,1,3,189
3,1,4,188
4,1,5,187
5,1,6,186
6,1,7,185
7,1,8,184
8,1,9,183
9,1,10,182


count    20631.000000
mean       107.807862
std         68.880990
min          0.000000
25%         51.000000
50%        103.000000
75%        155.000000
max        361.000000
Name: RUL, dtype: float64

In [4]:

# Get the last observed cycle for each test engine
last_cycles = test_df.groupby('unit_number')['cycle'].max()

# Map engine IDs (1–100) to their official RUL values
official_rul_by_engine = dict(
    zip(
        range(1, len(test_rul) + 1),
        test_rul['RUL']
    )
)

test_df['official_RUL'] = test_df['unit_number'].map(
    official_rul_by_engine
)

# Reconstruct RUL for every observed cycle
test_df['RUL'] = (
    test_df['official_RUL']
    + test_df['unit_number'].map(last_cycles)
    - test_df['cycle']
)

# Validate mapping before removing the temporary column
assert test_df['official_RUL'].notna().all(), "Missing official RUL mapping!"
assert test_df['RUL'].notna().all(), "Missing reconstructed RUL values!"
assert (test_df['RUL'] >= 0).all(), "Negative RUL values found!"

test_df.drop(columns=['official_RUL'], inplace=True)

print("Test RUL generated successfully!")
print("Missing RUL values:", test_df['RUL'].isna().sum())

display(test_df[['unit_number', 'cycle', 'RUL']].head(10))


Test RUL generated successfully!
Missing RUL values: 0


,unit_number,cycle,RUL
0,1,1,142
1,1,2,141
2,1,3,140
3,1,4,139
4,1,5,138
5,1,6,137
6,1,7,136
7,1,8,135
8,1,9,134
9,1,10,133


In [5]:

# Get the final observed cycle for every test engine
final_rul = (
    test_df.sort_values(['unit_number', 'cycle'])
    .groupby('unit_number')
    .tail(1)
    .sort_values('unit_number')['RUL']
    .reset_index(drop=True)
)

# Official RUL values are ordered by engine ID (1–100)
expected_rul = test_rul['RUL'].reset_index(drop=True)

# Compare reconstructed and official RUL
matches = final_rul.equals(expected_rul)

print("Number of test engines:", len(final_rul))
print("All 100 engines verified:", matches)
print("Mismatches:", (final_rul != expected_rul).sum())

assert matches, "RUL verification failed!"
print("SUCCESS: All 100 engines match official RUL values.")


Number of test engines: 100
All 100 engines verified: True
Mismatches: 0
SUCCESS: All 100 engines match official RUL values.


In [6]:
from sklearn.model_selection import train_test_split

engine_ids = train_df['unit_number'].unique()

train_ids, val_ids = train_test_split(
    engine_ids,
    test_size=0.20,
    random_state=42
)

train_prepared = train_df[
    train_df['unit_number'].isin(train_ids)
].copy()

val_prepared = train_df[
    train_df['unit_number'].isin(val_ids)
].copy()

print("Training engines:", len(train_ids))
print("Validation engines:", len(val_ids))
print("Training rows:", train_prepared.shape)
print("Validation rows:", val_prepared.shape)

print("Common engines:",
      len(set(train_ids) & set(val_ids)))

Training engines: 80
Validation engines: 20
Training rows: (16561, 27)
Validation rows: (4070, 27)
Common engines: 0


In [7]:
sensor_cols = [f'sensor_{i:02d}' for i in range(1, 22)]

def create_features(df):
    df = df.sort_values(['unit_number', 'cycle']).copy()

    for col in sensor_cols:
        df[f'{col}_delta'] = df.groupby('unit_number')[col].diff().fillna(0)

    return df

train_prepared = create_features(train_prepared)
val_prepared = create_features(val_prepared)

# Isolate test input features from target RUL
test_features_input = test_df.drop(columns=['RUL'])
test_prepared = create_features(test_features_input)

# Preserve test labels separately
test_rul_expanded = test_df[['unit_number', 'cycle', 'RUL']].copy()

print("Feature engineering completed!")
print("Training shape:", train_prepared.shape)
print("Validation shape:", val_prepared.shape)
print("Test shape (features only):", test_prepared.shape)
print("Test RUL expanded shape:", test_rul_expanded.shape)
print("Is RUL in test_prepared?", 'RUL' in test_prepared.columns)

display(train_prepared.head())


Feature engineering completed!
Training shape: (16561, 48)
Validation shape: (4070, 48)
Test shape (features only): (13096, 47)
Test RUL expanded shape: (13096, 3)
Is RUL in test_prepared? False


,unit_number,cycle,setting_1,setting_2,setting_3,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL,sensor_01_delta,sensor_02_delta,sensor_03_delta,sensor_04_delta,sensor_05_delta,sensor_06_delta,sensor_07_delta,sensor_08_delta,sensor_09_delta,sensor_10_delta,sensor_11_delta,sensor_12_delta,sensor_13_delta,sensor_14_delta,sensor_15_delta,sensor_16_delta,sensor_17_delta,sensor_18_delta,sensor_19_delta,sensor_20_delta,sensor_21_delta
192,2,1,-0.0018,0.0006,100.0,518.67,641.89,1583.84,1391.28,14.62,21.60,554.53,2388.01,9054.72,1.3,46.93,522.33,2388.06,8137.72,8.3905,0.03,391,2388,100.0,38.94,23.4585,286,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0000,0.0,0.0,0.0,0.0,0.00,0.0000
193,2,2,0.0043,-0.0003,100.0,518.67,641.82,1587.05,1393.13,14.62,21.61,554.77,2387.98,9051.31,1.3,47.24,522.70,2387.98,8131.09,8.4167,0.03,392,2388,100.0,39.06,23.4085,285,0.0,-0.07,3.21,1.85,0.0,0.01,0.24,-0.03,-3.41,0.0,0.31,0.37,-0.08,-6.63,0.0262,0.0,1.0,0.0,0.0,0.12,-0.0500
194,2,3,0.0018,0.0003,100.0,518.67,641.55,1588.32,1398.96,14.62,21.60,555.14,2388.04,9054.24,1.3,47.22,522.58,2387.99,8140.58,8.3802,0.03,391,2388,100.0,39.11,23.4250,284,0.0,-0.27,1.27,5.83,0.0,-0.01,0.37,0.06,2.93,0.0,-0.02,-0.12,0.01,9.49,-0.0365,0.0,-1.0,0.0,0.0,0.05,0.0165
195,2,4,0.0035,-0.0004,100.0,518.67,641.68,1584.15,1396.08,14.62,21.61,554.25,2387.98,9058.01,1.3,47.10,522.49,2387.93,8140.44,8.4018,0.03,391,2388,100.0,39.13,23.5027,283,0.0,0.13,-4.17,-2.88,0.0,0.01,-0.89,-0.06,3.77,0.0,-0.12,-0.09,-0.06,-0.14,0.0216,0.0,0.0,0.0,0.0,0.02,0.0777
196,2,5,0.0005,0.0004,100.0,518.67,641.73,1579.03,1402.52,14.62,21.60,555.12,2388.03,9058.15,1.3,47.25,522.27,2387.94,8136.67,8.3867,0.03,390,2388,100.0,39.18,23.4234,282,0.0,0.05,-5.12,6.44,0.0,-0.01,0.87,0.05,0.14,0.0,0.15,-0.22,0.01,-3.77,-0.0151,0.0,-1.0,0.0,0.0,0.05,-0.0793


In [8]:
datasets = {
    "Train": train_prepared,
    "Validation": val_prepared,
    "Test": test_prepared
}

for name, df in datasets.items():
    print(f"\n{name} Dataset")
    print("Shape:", df.shape)
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate (unit, cycle) keys:", df.duplicated(subset=['unit_number', 'cycle']).sum())
    assert df.isnull().sum().sum() == 0, f"Missing values found in {name}!"
    assert not df.duplicated(subset=['unit_number', 'cycle']).any(), f"Duplicate keys in {name}!"

# Verify target values
print("\nTarget RUL Validation:")
assert (train_prepared['RUL'] >= 0).all(), "Negative RUL in train_prepared!"
assert (val_prepared['RUL'] >= 0).all(), "Negative RUL in val_prepared!"
assert (test_rul_expanded['RUL'] >= 0).all(), "Negative RUL in test_rul_expanded!"
assert 'RUL' not in test_prepared.columns, "RUL leaked into test_prepared input features!"
print("Negative RUL count (Train, Val, Test Expanded): 0, 0, 0")
print("RUL excluded from test input features: True")

# Verify engine-wise split isolation
train_engines = set(train_prepared['unit_number'])
val_engines = set(val_prepared['unit_number'])
train_val_overlap = train_engines & val_engines

print("\nEngine Namespace & Split Isolation:")
print("Train engines count:", len(train_engines))
print("Validation engines count:", len(val_engines))
print("Train-Validation engine overlap:", len(train_val_overlap))
assert len(train_val_overlap) == 0, f"Train-validation engine leakage detected: {train_val_overlap}!"

# Note: Train engines (1-100) and Test engines (1-100) belong to independent C-MAPSS dataset namespaces.
# They are physically distinct engines that share the same 1-100 ID sequence; only train-val overlap must be zero.

# Verify test feature/label alignment
assert len(test_prepared) == len(test_rul_expanded), "Test feature and label row count mismatch!"
assert (test_prepared['unit_number'].to_numpy() == test_rul_expanded['unit_number'].to_numpy()).all(), "Engine ID mismatch between test features and labels!"
assert (test_prepared['cycle'].to_numpy() == test_rul_expanded['cycle'].to_numpy()).all(), "Cycle mismatch between test features and labels!"
print("Test features and labels row count & key alignment: True")



Train Dataset
Shape: (16561, 48)
Missing values: 0
Duplicate (unit, cycle) keys: 0

Validation Dataset
Shape: (4070, 48)
Missing values: 0
Duplicate (unit, cycle) keys: 0

Test Dataset
Shape: (13096, 47)
Missing values: 0
Duplicate (unit, cycle) keys: 0

Target RUL Validation:
Negative RUL count (Train, Val, Test Expanded): 0, 0, 0
RUL excluded from test input features: True

Engine Namespace & Split Isolation:
Train engines count: 80
Validation engines count: 20
Train-Validation engine overlap: 0
Test features and labels row count & key alignment: True


In [9]:
import json

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# 1. Export training and validation prepared datasets (features + target)
train_prepared.to_csv(
    PROCESSED_DATA_DIR / "train_prepared.csv", index=False
)

val_prepared.to_csv(
    PROCESSED_DATA_DIR / "validation_prepared.csv", index=False
)

# 2. Export test prepared dataset (features only, strictly NO RUL)
test_prepared.to_csv(
    PROCESSED_DATA_DIR / "test_prepared.csv", index=False
)

# 3. Export expanded test RUL labels (aligned per engine and cycle)
test_rul_expanded.to_csv(
    PROCESSED_DATA_DIR / "test_rul_expanded.csv", index=False
)

# 4. Export official test RUL labels (ordered by unit_number)
test_rul_official = pd.DataFrame({
    "unit_number": np.arange(1, len(test_rul) + 1),
    "RUL": test_rul["RUL"].to_numpy()
})

test_rul_official.to_csv(
    PROCESSED_DATA_DIR / "test_rul_official.csv", index=False
)

# 5. Define feature columns (excluding unit_number identifier and RUL target)
feature_columns = [
    col for col in train_prepared.columns if col not in ["unit_number", "RUL"]
]

# 6. Construct unified split manifest
manifest = {
    "dataset": "NASA C-MAPSS FD001",
    "random_state": 42,
    "split_strategy": "Engine-wise 80:20 split",
    "train_engines": sorted(map(int, train_ids)),
    "validation_engines": sorted(map(int, val_ids)),
    "train_rows": len(train_prepared),
    "validation_rows": len(val_prepared),
    "test_rows": len(test_prepared),
    "feature_columns": feature_columns,
    "target_column": "RUL",
    "sensor_columns": sensor_cols,
    "delta_feature_columns": [f"{col}_delta" for col in sensor_cols],
    "feature_engineering": [
        "Original sensor columns zero-padded (sensor_01 to sensor_21) for backend schema consistency",
        "Cycle-to-cycle sensor deltas added (sensor_01_delta to sensor_21_delta)",
        "cycle_norm excluded to avoid future information leakage",
        "unit_number excluded from model feature columns",
        "cycle retained as a model input feature"
    ]
}

with open(PROCESSED_DATA_DIR / "split_manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

print("All datasets and manifest saved successfully!")
print("Train shape:", train_prepared.shape)
print("Validation shape:", val_prepared.shape)
print("Test shape (features only):", test_prepared.shape)
print("Test RUL expanded shape:", test_rul_expanded.shape)
print("Test RUL official shape:", test_rul_official.shape)
print(sorted([f.name for f in PROCESSED_DATA_DIR.iterdir()]))


All datasets and manifest saved successfully!
Train shape: (16561, 48)
Validation shape: (4070, 48)
Test shape (features only): (13096, 47)
Test RUL expanded shape: (13096, 3)
Test RUL official shape: (100, 2)
['split_manifest.json', 'test_prepared.csv', 'test_rul_expanded.csv', 'test_rul_official.csv', 'train_prepared.csv', 'validation_prepared.csv']


In [10]:
DOCS_DIR.mkdir(parents=True, exist_ok=True)

documentation = """
# YantraDrishti - Day 2 Data Preparation

## Dataset
- Source: NASA C-MAPSS Turbofan Dataset
- Subset: FD001
- Training engines: 100
- Test engines: 100
- Training records: 20,631
- Test records: 13,096

## RUL Generation

Training:
RUL = Maximum cycle of engine - Current cycle

Testing:
RUL = Official RUL + (Last observed cycle - Current cycle)

Test RUL labels are stored separately from test input features.

## Train-Validation Split

- Training engines: 80
- Validation engines: 20
- Random state: 42
- Split performed by unit_number.
- No engine overlap between training and validation.

## Feature Engineering

- Canonical zero-padded sensor names are used (sensor_01 to sensor_21) to match backend schemas.
- Cycle-to-cycle sensor differences are added (sensor_01_delta to sensor_21_delta).
- Delta features are calculated separately for each engine.
- cycle_norm is excluded because it uses the maximum observed cycle (future information leakage).
- unit_number is an entity identifier, excluded from model feature columns.
- cycle is retained as a causal model input feature.

No physical meanings are assigned to anonymous sensor channels.

## Validation

- Missing values checked.
- Duplicate rows checked.
- Negative RUL checked.
- Engine overlap checked.

## Output Files

- train_prepared.csv
- validation_prepared.csv
- test_prepared.csv
- test_rul_expanded.csv
- test_rul_official.csv
- split_manifest.json

## Important Decisions

- No synthetic data used.
- No model training performed.
- No random row-wise splitting.
- No fitted scaler applied yet.
- Test input features and test RUL labels are stored separately (no RUL in test_prepared.csv).
- split_manifest.json contains the final feature and target schema (46 features, RUL target).

## Next Phase

Model training, preprocessing fit on training data only,
model evaluation, and inference pipeline development.
"""

with open(DOCS_DIR / "day2_preprocessing.md", "w") as f:
    f.write(documentation)

print("Documentation created successfully!")
print(sorted([f.name for f in DOCS_DIR.iterdir()]))


Documentation created successfully!
['day2_preprocessing.md']


In [11]:
import pandas as pd
import json

files = [
    "train_prepared.csv",
    "validation_prepared.csv",
    "test_prepared.csv",
    "test_rul_expanded.csv",
    "test_rul_official.csv",
    "split_manifest.json"
]

for file in files:
    path = PROCESSED_DATA_DIR / file
    print(file, "Exists:", path.exists())
    assert path.exists(), f"Output file missing: {file}"

train_check = pd.read_csv(PROCESSED_DATA_DIR / "train_prepared.csv")
val_check = pd.read_csv(PROCESSED_DATA_DIR / "validation_prepared.csv")
test_check = pd.read_csv(PROCESSED_DATA_DIR / "test_prepared.csv")
rul_check = pd.read_csv(PROCESSED_DATA_DIR / "test_rul_expanded.csv")
official_rul_check = pd.read_csv(PROCESSED_DATA_DIR / "test_rul_official.csv")

with open(PROCESSED_DATA_DIR / "split_manifest.json") as f:
    manifest_check = json.load(f)

print("\n--- Shape Verification ---")
print("Train shape:", train_check.shape)
print("Validation shape:", val_check.shape)
print("Test shape:", test_check.shape)
print("Test RUL expanded shape:", rul_check.shape)
print("Test RUL official shape:", official_rul_check.shape)

# Missing values checks
assert train_check.isnull().sum().sum() == 0, "Train contains missing values!"
assert val_check.isnull().sum().sum() == 0, "Validation contains missing values!"
assert test_check.isnull().sum().sum() == 0, "Test contains missing values!"
assert rul_check.isnull().sum().sum() == 0, "test_rul_expanded contains missing values!"
assert official_rul_check.isnull().sum().sum() == 0, "test_rul_official contains missing values!"

# Duplicate engine-cycle key checks
assert not train_check.duplicated(subset=['unit_number', 'cycle']).any(), "Duplicate keys in train_prepared!"
assert not val_check.duplicated(subset=['unit_number', 'cycle']).any(), "Duplicate keys in validation_prepared!"
assert not test_check.duplicated(subset=['unit_number', 'cycle']).any(), "Duplicate keys in test_prepared!"

# Negative RUL checks
assert (train_check['RUL'] >= 0).all(), "Negative RUL values in train_prepared!"
assert (val_check['RUL'] >= 0).all(), "Negative RUL values in validation_prepared!"
assert (rul_check['RUL'] >= 0).all(), "Negative RUL values in test_rul_expanded!"
assert (official_rul_check['RUL'] >= 0).all(), "Negative RUL values in test_rul_official!"

# Train-Validation engine overlap check
train_val_leak = set(train_check.unit_number) & set(val_check.unit_number)
assert len(train_val_leak) == 0, f"Train-validation overlap found: {train_val_leak}!"

# Test target isolation and feature schema check against manifest
assert 'RUL' not in test_check.columns, "RUL leaked into test_prepared.csv!"
expected_feature_cols = ['unit_number'] + manifest_check['feature_columns']
assert list(test_check.columns) == expected_feature_cols, "test_prepared.csv columns do not match manifest!"

# Test feature and label alignment
assert len(test_check) == len(rul_check), "Row count mismatch between test features and expanded labels!"
assert (test_check['unit_number'].to_numpy() == rul_check['unit_number'].to_numpy()).all(), "Engine ID alignment mismatch!"
assert (test_check['cycle'].to_numpy() == rul_check['cycle'].to_numpy()).all(), "Cycle alignment mismatch!"

# Official RUL verification
assert list(official_rul_check.columns) == ['unit_number', 'RUL'], "test_rul_official columns must be ['unit_number', 'RUL']!"
assert len(official_rul_check) == 100, f"Expected 100 official test engines, got {len(official_rul_check)}!"

print("\nALL POST-EXPORT VALIDATION ASSERTIONS PASSED SUCCESSFULLY!")


train_prepared.csv Exists: True
validation_prepared.csv Exists: True
test_prepared.csv Exists: True
test_rul_expanded.csv Exists: True
test_rul_official.csv Exists: True
split_manifest.json Exists: True

--- Shape Verification ---
Train shape: (16561, 48)
Validation shape: (4070, 48)
Test shape: (13096, 47)
Test RUL expanded shape: (13096, 3)
Test RUL official shape: (100, 2)

ALL POST-EXPORT VALIDATION ASSERTIONS PASSED SUCCESSFULLY!
